In [1]:
# ============================================================
# Cell 1 — Load 4 per-robot CSVs (independent runs)
# ============================================================
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 260)

FILES = {
    1: "dataset_raw_static_square_r1_10min.csv",
    2: "dataset_raw_static_square_r2_10min.csv",
    3: "dataset_raw_static_square_r3_10min.csv",
    4: "dataset_raw_static_square_r4_10min.csv",
}

# ---------- LOAD ----------
frames = []
for src, fname in FILES.items():
    p = Path(fname)
    if not p.exists():
        print(f"  MISSING: {fname}")
        continue
    d = pd.read_csv(p)
    d["source_robot"] = src
    frames.append(d)
    print(f"  loaded {fname:52s}  rows={len(d):>6d}  cols={d.shape[1]}")

df = pd.concat(frames, ignore_index=True)
print(f"\n  merged rows: {len(df)}")

# ---------- LINK STRUCTURE PER SOURCE ----------
print("\n" + "=" * 88)
print(" LINK STRUCTURE PER SOURCE ROBOT")
print("=" * 88)
print(f"{'src':>4s} | {'init -> resp':>14s} | {'n':>6s} | {'frames':>7s} | "
      f"{'dur[s]':>8s} | {'mean_cfo_dist':>14s} | {'std_cfo_dist':>13s}")
print("-" * 88)

for src in sorted(df["source_robot"].unique()):
    sub = df[df["source_robot"] == src]
    t_min = sub["timestamp_ms"].min()
    t_max = sub["timestamp_ms"].max()
    dur = (t_max - t_min) / 1000.0
    for (i, j), g in sub.groupby(["init_id", "resp_id"]):
        print(f"{src:>4d} | {i:>5d} -> {j:<5d} | {len(g):>6d} | "
              f"{g['frame_id'].nunique():>7d} | {dur:>8.1f} | "
              f"{g['dist_cfo_m'].mean():>14.4f} | "
              f"{g['dist_cfo_m'].std():>13.4f}")

# ---------- COLUMN CONSISTENCY ----------
print("\n" + "=" * 88)
print(" COLUMN CONSISTENCY CHECK")
print("=" * 88)
cols_by_src = {src: list(df[df["source_robot"] == src].columns)
               for src in sorted(df["source_robot"].unique())}
ref = cols_by_src[1]
for src, cols in cols_by_src.items():
    if cols == ref:
        print(f"  src {src}: identical to src 1 ({len(cols)} cols)")
    else:
        extra = set(cols) - set(ref)
        missing = set(ref) - set(cols)
        print(f"  src {src}: +{sorted(extra)}  -{sorted(missing)}")

# ---------- STATUS FILTER PREVIEW ----------
print("\n" + "=" * 88)
print(" STATUS DISTRIBUTION PER SOURCE")
print("=" * 88)
print(f"{'src':>4s} | {'total':>7s} | {'status=1':>9s} | {'status=0':>9s} | "
      f"{'drop%':>7s}")
print("-" * 52)
for src in sorted(df["source_robot"].unique()):
    sub = df[df["source_robot"] == src]
    n_tot = len(sub); n_ok = (sub["status"] == 1).sum()
    print(f"{src:>4d} | {n_tot:>7d} | {n_ok:>9d} | {n_tot-n_ok:>9d} | "
          f"{100.0*(n_tot-n_ok)/n_tot:>6.2f}%")

# ---------- TEMPERATURE / TIME OVERVIEW PER SOURCE ----------
print("\n" + "=" * 88)
print(" TEMPERATURE AND TIME SPAN PER SOURCE")
print("=" * 88)
print(f"{'src':>4s} | {'t_start':>10s} | {'t_end':>10s} | "
      f"{'T_UWB(min)':>10s} {'T_UWB(max)':>10s} | "
      f"{'T_ESP(min)':>10s} {'T_ESP(max)':>10s}")
print("-" * 88)
for src in sorted(df["source_robot"].unique()):
    sub = df[df["source_robot"] == src]
    t0 = sub["timestamp_ms"].min() / 1000.0
    t1 = sub["timestamp_ms"].max() / 1000.0
    Ti = sub["temp_uwb_init"]; Tr = sub["temp_uwb_resp"]
    T_all = pd.concat([Ti, Tr])
    Ei = sub["temp_esp_init"]; Er = sub["temp_esp_resp"]
    E_all = pd.concat([Ei, Er])
    print(f"{src:>4d} | {t0:>10.1f} | {t1:>10.1f} | "
          f"{T_all.min():>10.2f} {T_all.max():>10.2f} | "
          f"{E_all.min():>10.2f} {E_all.max():>10.2f}")

  loaded dataset_raw_static_square_r1_10min.csv                rows= 10755  cols=32
  loaded dataset_raw_static_square_r2_10min.csv                rows=  8916  cols=32
  loaded dataset_raw_static_square_r3_10min.csv                rows=  9456  cols=32
  loaded dataset_raw_static_square_r4_10min.csv                rows= 10260  cols=32

  merged rows: 39387

 LINK STRUCTURE PER SOURCE ROBOT
 src |   init -> resp |      n |  frames |   dur[s] |  mean_cfo_dist |  std_cfo_dist
----------------------------------------------------------------------------------------
   1 |     1 -> 2     |   3585 |    3585 |    717.0 |         1.6026 |        1.3953
   1 |     1 -> 3     |   3585 |    3585 |    717.0 |         7.3546 |        1.0479
   1 |     1 -> 4     |   3585 |    3585 |    717.0 |         2.4879 |        1.3833
   2 |     2 -> 1     |   2972 |    2972 |    595.9 |        80.1352 |        2.4998
   2 |     2 -> 3     |   2972 |    2972 |    595.9 |        45.6097 |        0.5034
   2 |   

In [2]:
# ============================================================
# Cell 2 — Filter invalid rows, build bidirectional link table
# ============================================================
# Notes:
# - Some rows may have zeros in temperature or distance due to 
#   connection loss or sensor read failure. Filter them out.
# - For each physical pair (i,j) we now have data from BOTH 
#   directions (i->j and j->i) because each robot logs independently.
# ============================================================

print("=" * 88)
print(" STEP 1: FILTER INVALID ROWS")
print("=" * 88)

n0 = len(df)
print(f"  initial rows: {n0}")

# --- Filter 1: status must be 1
df = df[df["status"] == 1].copy()
n1 = len(df)
print(f"  after status==1         : {n1:>6d}  (dropped {n0-n1})")

# --- Filter 2: distance must be positive and finite
mask_dist = (df["dist_cfo_m"] > 0.1) & np.isfinite(df["dist_cfo_m"])
df = df[mask_dist].copy()
n2 = len(df)
print(f"  after dist_cfo_m>0.1    : {n2:>6d}  (dropped {n1-n2})")

# --- Filter 3: temperatures must be in physical range
mask_ti = (df["temp_uwb_init"] > 10) & (df["temp_uwb_init"] < 80)
mask_tr = (df["temp_uwb_resp"] > 10) & (df["temp_uwb_resp"] < 80)
df = df[mask_ti & mask_tr].copy()
n3 = len(df)
print(f"  after temp range 10..80 : {n3:>6d}  (dropped {n2-n3})")

# --- Filter 4: RSSI must be in physical range
mask_rssi = (df["rssi_dbm"] > -100) & (df["rssi_dbm"] < -30)
df = df[mask_rssi].copy()
n4 = len(df)
print(f"  after RSSI range        : {n4:>6d}  (dropped {n3-n4})")

print(f"\n  final clean rows: {len(df)}")
print(f"  retention rate  : {100.0 * len(df) / n0:.2f}%")

# ---------- Build physical-pair key (i<j) for bidirectional comparison ----------
flip = df["init_id"] > df["resp_id"]
df["i_id"] = np.where(flip, df["resp_id"], df["init_id"]).astype(int)
df["j_id"] = np.where(flip, df["init_id"], df["resp_id"]).astype(int)
df["pair"] = df["i_id"].astype(str) + "-" + df["j_id"].astype(str)

# direction marker: "forward" if source_robot == i_id, else "reverse"
df["direction"] = np.where(df["source_robot"] == df["i_id"], "fwd", "rev")

# ---------- BIDIRECTIONAL LINK TABLE ----------
print("\n" + "=" * 96)
print(" STEP 2: BIDIRECTIONAL LINK ANALYSIS")
print("=" * 96)
print(" For each pair (i,j), compare dist_cfo_m from each direction.")
print(" Asymmetry = (rev - fwd) / 0.5*(rev + fwd)")
print(" Large asymmetry → transmitter/receiver chains differ between robots.")
print("-" * 96)
print(f"{'pair':>6s} | {'dir':>4s} | {'src':>4s} | {'n':>6s} | "
      f"{'mean_cfo':>11s} | {'std':>9s} | {'median':>9s} | "
      f"{'asym%':>8s}")
print("-" * 96)

# Collect statistics
pair_stats = {}
for pair, g in df.groupby("pair"):
    pair_stats[pair] = {}
    for direction, gg in g.groupby("direction"):
        src_set = sorted(gg["source_robot"].unique())
        pair_stats[pair][direction] = {
            "src": src_set[0] if len(src_set) == 1 else "mix",
            "n": len(gg),
            "mean": gg["dist_cfo_m"].mean(),
            "std": gg["dist_cfo_m"].std(),
            "median": gg["dist_cfo_m"].median(),
        }

# Print table
for pair in sorted(pair_stats.keys()):
    fwd = pair_stats[pair].get("fwd")
    rev = pair_stats[pair].get("rev")
    
    if fwd:
        print(f"{pair:>6s} | fwd  | {fwd['src']:>4} | {fwd['n']:>6d} | "
              f"{fwd['mean']:>11.4f} | {fwd['std']:>9.4f} | "
              f"{fwd['median']:>9.4f} |", end="")
    else:
        print(f"{pair:>6s} | fwd  |    - |      - | "
              f"{'-':>11s} | {'-':>9s} | {'-':>9s} |", end="")
    
    # asymmetry
    if fwd and rev:
        m_f, m_r = fwd["mean"], rev["mean"]
        asym = (m_r - m_f) / (0.5 * (m_r + m_f)) * 100
        print(f" {asym:>7.2f}%")
    else:
        print(f" {'-':>8s}")
    
    if rev:
        print(f"{'':>6s} | rev  | {rev['src']:>4} | {rev['n']:>6d} | "
              f"{rev['mean']:>11.4f} | {rev['std']:>9.4f} | "
              f"{rev['median']:>9.4f} |")
    else:
        print(f"{'':>6s} | rev  |    - |      - | "
              f"{'-':>11s} | {'-':>9s} | {'-':>9s} |")
    print("-" * 96)

# ---------- SUMMARY TABLE (one row per pair) ----------
print("\n" + "=" * 96)
print(" SUMMARY: ASYMMETRY PER PHYSICAL PAIR")
print("=" * 96)
print(f"{'pair':>6s} | {'mean_fwd':>10s} | {'mean_rev':>10s} | "
      f"{'diff [m]':>10s} | {'asym %':>9s} | {'std_fwd':>9s} | {'std_rev':>9s}")
print("-" * 96)

summary_rows = []
for pair in sorted(pair_stats.keys()):
    fwd = pair_stats[pair].get("fwd")
    rev = pair_stats[pair].get("rev")
    if not (fwd and rev):
        continue
    diff = rev["mean"] - fwd["mean"]
    asym = diff / (0.5 * (rev["mean"] + fwd["mean"])) * 100
    print(f"{pair:>6s} | {fwd['mean']:>10.4f} | {rev['mean']:>10.4f} | "
          f"{diff:>+10.4f} | {asym:>8.2f}% | "
          f"{fwd['std']:>9.4f} | {rev['std']:>9.4f}")
    summary_rows.append({
        "pair": pair,
        "mean_fwd": fwd["mean"], "mean_rev": rev["mean"],
        "diff_m": diff, "asym_pct": asym,
        "std_fwd": fwd["std"], "std_rev": rev["std"],
        "src_fwd": fwd["src"], "src_rev": rev["src"],
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv("output_bidirectional_pairs.csv", index=False)
print(f"\n  Saved: output_bidirectional_pairs.csv")

# ---------- CLEAN DATA SAVE ----------
df_clean = df.drop(columns=["i_id", "j_id"]).copy()
df_clean.to_csv("output_clean_merged.csv", index=False)
print(f"  Saved: output_clean_merged.csv  ({len(df_clean)} rows)")

 STEP 1: FILTER INVALID ROWS
  initial rows: 39387
  after status==1         :  39362  (dropped 25)
  after dist_cfo_m>0.1    :  39325  (dropped 37)
  after temp range 10..80 :  39325  (dropped 0)
  after RSSI range        :  39322  (dropped 3)

  final clean rows: 39322
  retention rate  : 99.83%

 STEP 2: BIDIRECTIONAL LINK ANALYSIS
 For each pair (i,j), compare dist_cfo_m from each direction.
 Asymmetry = (rev - fwd) / 0.5*(rev + fwd)
 Large asymmetry → transmitter/receiver chains differ between robots.
------------------------------------------------------------------------------------------------
  pair |  dir |  src |      n |    mean_cfo |       std |    median |    asym%
------------------------------------------------------------------------------------------------
   1-2 | fwd  |    1 |   3544 |      1.6203 |    1.3937 |    1.1612 |  192.08%
       | rev  |    2 |   2970 |     80.1892 |    1.3873 |   80.6144 |
------------------------------------------------------------------

In [3]:
# ============================================================
# Cell 3 — Extract per-robot initiator bias (D_i) and test consistency
# ============================================================
# Theory: in SS-TWR, d_measured = d_true + D_initiator
# where D_i depends only on the initiator's TX+RX antenna delays.
# 
# Test: If the model holds, each robot's D_i must be constant across
# its three outgoing links.
# ============================================================

GT = {("1","2"): 2.000, ("1","3"): 2.828, ("1","4"): 2.000,
      ("2","3"): 2.000, ("2","4"): 2.828, ("3","4"): 2.000}
GT = {tuple(sorted(k)): v for k, v in GT.items()}

# Use dist_raw_m (no CFO compensation yet) to see the raw bias
# Then repeat with dist_cfo_m to see effect of CFO comp
print("=" * 96)
print(" STEP 1: RAW DISTANCE PER DIRECTION  (no compensation)")
print("=" * 96)
print(f"{'pair':>6s} | {'init':>4s} | {'resp':>4s} | {'n':>5s} | "
      f"{'GT [m]':>7s} | {'d_meas':>8s} | {'D_init':>8s} | "
      f"{'std [m]':>8s}")
print("-" * 96)

raw_bias = {}
for (init, resp), g in df.groupby(["init_id", "resp_id"]):
    pair = tuple(sorted([str(init), str(resp)]))
    if pair not in GT:
        continue
    d_meas = g["dist_raw_m"].mean()
    d_true = GT[pair]
    D_init = d_meas - d_true
    print(f"{pair[0]}-{pair[1]:>3s} | {init:>4d} | {resp:>4d} | {len(g):>5d} | "
          f"{d_true:>7.3f} | {d_meas:>8.3f} | {D_init:>+8.3f} | "
          f"{g['dist_raw_m'].std():>8.3f}")
    raw_bias[(init, resp)] = D_init

print("\n" + "=" * 96)
print(" STEP 2: CONSISTENCY CHECK — Is D_init constant per robot?")
print("=" * 96)
print(" If yes → the model d = d_true + D_initiator is valid.")
print(" If no  → additional per-link or per-responder term needed.")
print("-" * 96)

for robot in sorted(df["init_id"].unique()):
    sub = [(k[1], v) for k, v in raw_bias.items() if k[0] == robot]
    if len(sub) == 0:
        continue
    Ds = [v for _, v in sub]
    spread = max(Ds) - min(Ds)
    flag = "  ⚠ inconsistent" if spread > 0.5 else "  ✓ consistent"
    print(f"\n  Robot {robot} as initiator:")
    for resp, D in sub:
        print(f"    → to R{resp}: D = {D:+.3f} m")
    print(f"    spread = {spread:.3f} m{flag}")

# ---------- Repeat with dist_cfo_m ----------
print("\n" + "=" * 96)
print(" STEP 3: SAME ANALYSIS WITH CFO-COMPENSATED DISTANCE")
print("=" * 96)
print(f"{'pair':>6s} | {'init':>4s} | {'resp':>4s} | {'n':>5s} | "
      f"{'GT [m]':>7s} | {'d_meas':>8s} | {'D_init':>8s} | "
      f"{'std [m]':>8s}")
print("-" * 96)

cfo_bias = {}
for (init, resp), g in df.groupby(["init_id", "resp_id"]):
    pair = tuple(sorted([str(init), str(resp)]))
    if pair not in GT:
        continue
    d_meas = g["dist_cfo_m"].mean()
    d_true = GT[pair]
    D_init = d_meas - d_true
    print(f"{pair[0]}-{pair[1]:>3s} | {init:>4d} | {resp:>4d} | {len(g):>5d} | "
          f"{d_true:>7.3f} | {d_meas:>8.3f} | {D_init:>+8.3f} | "
          f"{g['dist_cfo_m'].std():>8.3f}")
    cfo_bias[(init, resp)] = D_init

print("\n  Per-robot consistency (CFO-compensated):")
for robot in sorted(df["init_id"].unique()):
    sub = [(k[1], v) for k, v in cfo_bias.items() if k[0] == robot]
    if len(sub) == 0:
        continue
    Ds = [v for _, v in sub]
    spread = max(Ds) - min(Ds)
    flag = "  ⚠ inconsistent" if spread > 0.5 else "  ✓ consistent"
    print(f"\n  Robot {robot} as initiator:")
    for resp, D in sub:
        print(f"    → to R{resp}: D = {D:+.3f} m")
    print(f"    spread = {spread:.3f} m{flag}")

# ---------- Save ----------
result = []
for (init, resp), _ in df.groupby(["init_id", "resp_id"]):
    pair = tuple(sorted([str(init), str(resp)]))
    if pair not in GT:
        continue
    result.append({
        "pair": f"{pair[0]}-{pair[1]}",
        "initiator": init,
        "responder": resp,
        "gt_m": GT[pair],
        "D_init_raw": raw_bias[(init, resp)],
        "D_init_cfo": cfo_bias[(init, resp)],
    })
pd.DataFrame(result).to_csv("output_initiator_bias.csv", index=False)
print("\n  Saved: output_initiator_bias.csv")

 STEP 1: RAW DISTANCE PER DIRECTION  (no compensation)
  pair | init | resp |     n |  GT [m] |   d_meas |   D_init |  std [m]
------------------------------------------------------------------------------------------------
1-  2 |    1 |    2 |  3544 |   2.000 |   21.242 |  +19.242 |    0.683
1-  3 |    1 |    3 |  3585 |   2.828 |   24.481 |  +21.653 |    0.509
1-  4 |    1 |    4 |  3585 |   2.000 |   21.541 |  +19.541 |    0.681
1-  2 |    2 |    1 |  2970 |   2.000 |   60.533 |  +58.533 |    0.708
2-  3 |    2 |    3 |  2972 |   2.000 |   43.087 |  +41.087 |    0.270
2-  4 |    2 |    4 |  2972 |   2.828 |   42.223 |  +39.395 |    0.043
1-  3 |    3 |    1 |  3146 |   2.828 |   58.585 |  +55.757 |    0.387
2-  3 |    3 |    2 |  3152 |   2.000 |   38.057 |  +36.057 |    0.210
3-  4 |    3 |    4 |  3151 |   2.000 |   39.084 |  +37.084 |    0.222
1-  4 |    4 |    1 |  3407 |   2.000 |   59.673 |  +57.673 |    0.693
2-  4 |    4 |    2 |  3420 |   2.828 |   41.031 |  +38.203 |    0

In [5]:
# ============================================================
# Cell 4b — CORRECTED fit: d = d_true + A_i + B_j + c
# ============================================================
# Previous gauge sum(A)+sum(B)=0 was invariant → invalid.
# Fix: 9-param model with explicit constant c,
#      gauge: sum(A)=0, sum(B)=0.
# ============================================================

def fit_ABc_model(df_sub, dist_col):
    # Build observation matrix: 12 rows (6 pairs × 2 directions)
    obs = []
    for (init, resp), g in df_sub.groupby(["init_id", "resp_id"]):
        pair = tuple(sorted([str(init), str(resp)]))
        if pair not in GT:
            continue
        bias = g[dist_col].mean() - GT[pair]
        obs.append((int(init), int(resp), bias))
    
    devices = sorted(set([o[0] for o in obs]) | set([o[1] for o in obs]))
    idx = {d: k for k, d in enumerate(devices)}
    
    # Design matrix: [A_i, B_j, 1]
    A_mat = np.zeros((len(obs), 9))
    y = np.zeros(len(obs))
    for k, (init, resp, bias) in enumerate(obs):
        A_mat[k, idx[init]] = 1.0
        A_mat[k, 4 + idx[resp]] = 1.0
        A_mat[k, 8] = 1.0
        y[k] = bias
    
    # Gauge: sum(A) = 0, sum(B) = 0
    g1 = np.zeros(9); g1[:4] = 1.0
    g2 = np.zeros(9); g2[4:8] = 1.0
    A_aug = np.vstack([A_mat, g1, g2])
    y_aug = np.concatenate([y, [0.0, 0.0]])
    
    coef, *_ = np.linalg.lstsq(A_aug, y_aug, rcond=None)
    A_vals = coef[:4]
    B_vals = coef[4:8]
    c_val = coef[8]
    
    pred = A_mat @ coef
    resid = y - pred
    
    return {
        "devices": devices,
        "A": A_vals, "B": B_vals, "c": c_val,
        "obs": obs, "y": y, "pred": pred, "resid": resid,
        "rmse": np.sqrt(np.mean(resid**2)),
        "max_abs_resid": np.max(np.abs(resid)),
    }


print("=" * 92)
print(" CORRECTED MODEL FIT:  y(i→j) = A_i + B_j + c")
print("=" * 92)

for label, col in [("RAW", "dist_raw_m"), ("CFO", "dist_cfo_m")]:
    res = fit_ABc_model(df, col)
    print(f"\n[{label}]  c = {res['c']:+.4f} m    RMSE = {res['rmse']*100:.2f} cm    "
          f"max_resid = {res['max_abs_resid']*100:.2f} cm")
    print(f"{'robot':>5s} | {'A_i [m]':>10s} | {'B_j [m]':>10s} | "
          f"{'A_i + c/2':>10s} | {'B_j + c/2':>10s}")
    print("-" * 60)
    for k, d in enumerate(res["devices"]):
        print(f"{d:>5d} | {res['A'][k]:>+10.4f} | {res['B'][k]:>+10.4f} | "
              f"{res['A'][k] + res['c']/2:>+10.4f} | "
              f"{res['B'][k] + res['c']/2:>+10.4f}")

# Show residuals per pair
res = fit_ABc_model(df, "dist_raw_m")
print("\n  Per-pair residuals (RAW):")
print(f"  {'init':>4s} {'resp':>4s} | {'observed':>10s} | "
      f"{'predicted':>10s} | {'resid [cm]':>10s}")
print("  " + "-" * 52)
for k, (init, resp, bias) in enumerate(res["obs"]):
    print(f"  {init:>4d} {resp:>4d} | {bias:>+10.4f} | "
          f"{res['pred'][k]:>+10.4f} | {res['resid'][k]*100:>+10.2f}")

out = pd.DataFrame({
    "device": res["devices"],
    "A_raw_m": res["A"],
    "B_raw_m": res["B"],
    "c_raw_m": res["c"],
})
out.to_csv("output_ABc_corrected.csv", index=False)
print("\n  Saved: output_ABc_corrected.csv")

 CORRECTED MODEL FIT:  y(i→j) = A_i + B_j + c

[RAW]  c = +38.7532 m    RMSE = 13.90 cm    max_resid = 21.58 cm
robot |    A_i [m] |    B_j [m] |  A_i + c/2 |  B_j + c/2
------------------------------------------------------------
    1 |   -13.9711 |   +13.9106 |    +5.4055 |   +33.2872
    2 |    +5.6888 |    -5.6897 |   +25.0654 |   +13.6869
    3 |    +3.1511 |    -3.1848 |   +22.5277 |   +16.1918
    4 |    +5.1312 |    -5.0360 |   +24.5078 |   +14.3406

[CFO]  c = +38.7449 m    RMSE = 14.92 cm    max_resid = 26.62 cm
robot |    A_i [m] |    B_j [m] |  A_i + c/2 |  B_j + c/2
------------------------------------------------------------
    1 |   -27.9230 |   +27.8311 |    -8.5505 |   +47.2036
    2 |   +11.4098 |   -11.3441 |   +30.7823 |    +8.0283
    3 |    +6.2844 |    -6.3804 |   +25.6569 |   +12.9921
    4 |   +10.2288 |   -10.1066 |   +29.6013 |    +9.2659

  Per-pair residuals (RAW):
  init resp |   observed |  predicted | resid [cm]
  --------------------------------------

In [8]:
# ============================================================
# Cell 5 — Temporal stability of bias per link (self-contained)
# ============================================================

# ---- Rebuild essential columns if missing ----
if "t" not in df.columns:
    df["t"] = (df["timestamp_ms"] - df["timestamp_ms"].min()) / 1000.0

if "pair" not in df.columns:
    flip_ = df["init_id"] > df["resp_id"]
    df["i_id"] = np.where(flip_, df["resp_id"], df["init_id"]).astype(int)
    df["j_id"] = np.where(flip_, df["init_id"], df["resp_id"]).astype(int)
    df["pair"] = df["i_id"].astype(str) + "-" + df["j_id"].astype(str)

GT_pairs = {("1","2"): 2.000, ("1","3"): 2.828, ("1","4"): 2.000,
            ("2","3"): 2.000, ("2","4"): 2.828, ("3","4"): 2.000}
GT_pairs = {tuple(sorted(k)): v for k, v in GT_pairs.items()}

N_WINDOWS = 12

print("=" * 96)
print(" TEMPORAL STABILITY OF BIAS PER LINK  (within each source run)")
print("=" * 96)
print(" Bias = dist_cfo_m − GT  (in cm). Each row = one outgoing link.")
print("-" * 96)

drift_results = {}
for src in sorted(df["source_robot"].unique()):
    sub = df[df["source_robot"] == src].copy()
    if len(sub) < 100:
        continue
    t0, t1 = sub["t"].min(), sub["t"].max()
    edges = np.linspace(t0, t1, N_WINDOWS + 1)
    
    print(f"\n  Source robot {src}  (t = {t0:.0f} .. {t1:.0f} s):")
    print(f"    {'link':>7s} | " + " ".join(f"w{w:<2d}" for w in range(N_WINDOWS)) + " | drift")
    print("    " + "-" * (11 + 4 * N_WINDOWS + 10))
    
    for (init, resp), g in sub.groupby(["init_id", "resp_id"]):
        pair = tuple(sorted([str(init), str(resp)]))
        if pair not in GT_pairs:
            continue
        bias_series = []
        for w in range(N_WINDOWS):
            mask = (g["t"] >= edges[w]) & (g["t"] < edges[w + 1])
            if mask.sum() < 5:
                bias_series.append(np.nan)
                continue
            b = (g.loc[mask, "dist_cfo_m"].mean() - GT_pairs[pair]) * 100
            bias_series.append(b)
        valid = [b for b in bias_series if not np.isnan(b)]
        drift = max(valid) - min(valid) if len(valid) > 1 else 0.0
        
        print(f"    {pair[0]}-{pair[1]:>4s} | ", end="")
        for b in bias_series:
            if np.isnan(b):
                print(f"{'--':>4s}", end=" ")
            else:
                print(f"{b:>+4.0f}", end=" ")
        print(f" | {drift:>+6.1f} cm")
        
        drift_results[(src, f"{pair[0]}-{pair[1]}")] = {
            "series": bias_series,
            "drift": drift,
        }

# ---------- Cross-run comparison ----------
print("\n" + "=" * 96)
print(" CROSS-RUN COMPARISON")
print("=" * 96)
print(f"{'src':>4s} | {'t_start':>8s} | {'t_end':>8s} | "
      f"{'T_UWB mean':>11s} | {'T_ESP mean':>11s} | {'mean_bias':>10s}")
print("-" * 68)

for src in sorted(df["source_robot"].unique()):
    sub = df[df["source_robot"] == src]
    if len(sub) < 100:
        continue
    T_u = pd.concat([sub["temp_uwb_init"], sub["temp_uwb_resp"]]).mean()
    T_e = pd.concat([sub["temp_esp_init"], sub["temp_esp_resp"]]).mean()
    t0 = sub["timestamp_ms"].min() / 1000
    t1 = sub["timestamp_ms"].max() / 1000
    
    biases = []
    for (init, resp), g in sub.groupby(["init_id", "resp_id"]):
        pair = tuple(sorted([str(init), str(resp)]))
        if pair not in GT_pairs:
            continue
        biases.append(g["dist_cfo_m"].mean() - GT_pairs[pair])
    mean_bias = np.mean(biases) if biases else np.nan
    
    print(f"{src:>4d} | {t0:>8.0f} | {t1:>8.0f} | "
          f"{T_u:>11.2f} | {T_e:>11.2f} | {mean_bias:>+10.4f}")

# ---------- Save ----------
drift_data = []
for (src, pair), info in drift_results.items():
    drift_data.append({
        "source_robot": src,
        "pair": pair,
        "drift_cm": info["drift"],
    })
pd.DataFrame(drift_data).to_csv("output_temporal_drift.csv", index=False)
print(f"\n  Saved: output_temporal_drift.csv")

# ---------- Verdict ----------
print("\n" + "=" * 96)
print(" INTERPRETATION")
print("=" * 96)

for src in sorted(df["source_robot"].unique()):
    drifts = [v["drift"] for k, v in drift_results.items() if k[0] == src]
    if not drifts:
        continue
    mean_d = np.mean(drifts)
    max_d = max(drifts)
    if max_d < 30:
        verdict = "STABLE (drift < 30 cm)"
    elif max_d < 80:
        verdict = "moderate drift (30–80 cm)"
    else:
        verdict = "⚠ LARGE drift (> 80 cm)"
    print(f"  Source {src}: mean = {mean_d:+.1f} cm, "
          f"max = {max_d:+.1f} cm   →  {verdict}")

 TEMPORAL STABILITY OF BIAS PER LINK  (within each source run)
 Bias = dist_cfo_m − GT  (in cm). Each row = one outgoing link.
------------------------------------------------------------------------------------------------

  Source robot 1  (t = 0 .. 717 s):
       link | w0  w1  w2  w3  w4  w5  w6  w7  w8  w9  w10 w11 | drift
    ---------------------------------------------------------------------
    1-   2 | +257 +155  +96  +27  -48 -109 -156 -183 -183 -156 -116  -55  | +440.3 cm
    1-   3 | +637 +607 +574 +522 +456 +403 +360 +335 +335 +355 +391 +457  | +301.9 cm
    1-   4 | +329 +249 +191 +126  +43  -21  -68  -96  -95  -69  -33  +28  | +425.3 cm

  Source robot 2  (t = 0 .. 596 s):
       link | w0  w1  w2  w3  w4  w5  w6  w7  w8  w9  w10 w11 | drift
    ---------------------------------------------------------------------
    1-   2 | +7566 +7632 +7671 +7717 +7772 +7832 +7879 +7907 +7943 +7967 +7973 +7968  | +407.8 cm
    2-   3 | +4229 +4301 +4332 +4352 +4366 +4376 +4384 +43

In [9]:
# ============================================================
# Cell 6 — Separate thermal sensitivity of initiator vs responder
# ============================================================
# For each directed link (i→j), fit:
#     bias(t) = a0 + a1 * (T_i(t) - T_i_mean) + a2 * (T_j(t) - T_j_mean)
# where bias = dist_cfo_m - GT.
# Report a1, a2 with SE and p-values.
# ============================================================
from scipy.stats import t as t_dist

def ols_pvals(X, y):
    """OLS with intercept. Returns beta, SE, p-values, R²."""
    Xc = np.hstack([np.ones((len(X), 1)), X])
    beta, *_ = np.linalg.lstsq(Xc, y, rcond=None)
    y_pred = Xc @ beta
    resid = y - y_pred
    n, p = Xc.shape
    dof = max(n - p, 1)
    sigma2 = np.sum(resid ** 2) / dof
    try:
        cov_b = sigma2 * np.linalg.inv(Xc.T @ Xc)
        se = np.sqrt(np.diag(cov_b))
    except np.linalg.LinAlgError:
        se = np.full(p, np.nan)
    t_stat = np.where(se > 0, beta / se, 0.0)
    p_vals = 2 * (1 - t_dist.cdf(np.abs(t_stat), df=dof))
    ss_res = np.sum(resid ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
    return beta, se, p_vals, r2

GT_pairs = {("1","2"): 2.000, ("1","3"): 2.828, ("1","4"): 2.000,
            ("2","3"): 2.000, ("2","4"): 2.828, ("3","4"): 2.000}
GT_pairs = {tuple(sorted(k)): v for k, v in GT_pairs.items()}

print("=" * 100)
print(" DUAL TEMPERATURE REGRESSION PER DIRECTED LINK")
print(" Model: bias(t) = a0 + a1·(T_init - T̄_init) + a2·(T_resp - T̄_resp)")
print(" Units: a1, a2 in cm/°C.  bias in cm.")
print("=" * 100)
print(f"{'src':>3s} {'link':>7s} | {'n':>5s} | {'a1_init':>9s} {'p':>7s} | "
      f"{'a2_resp':>9s} {'p':>7s} | {'R²':>6s} | "
      f"{'corr_init':>10s} {'corr_resp':>10s} | {'corr_ii_jj':>10s}")
print("-" * 100)

regression_results = []

for src in sorted(df["source_robot"].unique()):
    sub = df[df["source_robot"] == src].copy()
    if len(sub) < 100:
        continue
    for (init, resp), g in sub.groupby(["init_id", "resp_id"]):
        pair = tuple(sorted([str(init), str(resp)]))
        if pair not in GT_pairs:
            continue
        g = g.sort_values("t").reset_index(drop=True)
        
        # Bias series (cm)
        bias = (g["dist_cfo_m"].values - GT_pairs[pair]) * 100.0
        
        # Temperature series
        T_i = g["temp_uwb_init"].values
        T_j = g["temp_uwb_resp"].values
        
        # Center
        T_i_c = T_i - T_i.mean()
        T_j_c = T_j - T_j.mean()
        
        X = np.column_stack([T_i_c, T_j_c])
        beta, se, pvals, r2 = ols_pvals(X, bias)
        
        a0, a1, a2 = beta
        p1, p2 = pvals[1], pvals[2]
        
        c_i = np.corrcoef(bias, T_i)[0, 1]
        c_j = np.corrcoef(bias, T_j)[0, 1]
        c_ij = np.corrcoef(T_i, T_j)[0, 1]
        
        # Significance markers
        s1 = "*" if p1 < 0.01 else ("" if p1 < 0.05 else "n.s.")
        s2 = "*" if p2 < 0.01 else ("" if p2 < 0.05 else "n.s.")
        
        print(f"{src:>3d} {pair[0]}-{pair[1]:>5s} | {len(g):>5d} | "
              f"{a1:>+9.2f} {s1:>7s} | "
              f"{a2:>+9.2f} {s2:>7s} | "
              f"{r2:>6.3f} | "
              f"{c_i:>+10.3f} {c_j:>+10.3f} | {c_ij:>+10.3f}")
        
        regression_results.append({
            "source": src,
            "init": init,
            "resp": resp,
            "pair": f"{pair[0]}-{pair[1]}",
            "direction": f"{init}->{resp}",
            "n": len(g),
            "a1_init_cm_per_C": a1,
            "a1_pvalue": p1,
            "a2_resp_cm_per_C": a2,
            "a2_pvalue": p2,
            "r2": r2,
            "corr_bias_init": c_i,
            "corr_bias_resp": c_j,
            "corr_init_resp": c_ij,
        })

# ---------- Save ----------
reg_df = pd.DataFrame(regression_results)
reg_df.to_csv("output_thermal_regression.csv", index=False)
print(f"\n  Saved: output_thermal_regression.csv")

# ---------- Aggregate by device role ----------
print("\n" + "=" * 100)
print(" AGGREGATED BY DEVICE ROLE")
print("=" * 100)

print("\n When a device is INITIATOR, its thermal coefficient (a1):")
print(f"{'device':>6s} | {'mean_a1':>10s} | {'std_a1':>10s} | {'n_links':>8s}")
print("-" * 48)
for d in sorted(df["init_id"].unique()):
    sub = reg_df[reg_df["init"] == d]
    if len(sub) == 0:
        continue
    print(f"{d:>6d} | {sub['a1_init_cm_per_C'].mean():>+10.2f} | "
          f"{sub['a1_init_cm_per_C'].std():>10.2f} | {len(sub):>8d}")

print("\n When a device is RESPONDER, its thermal coefficient (a2):")
print(f"{'device':>6s} | {'mean_a2':>10s} | {'std_a2':>10s} | {'n_links':>8s}")
print("-" * 48)
for d in sorted(df["resp_id"].unique()):
    sub = reg_df[reg_df["resp"] == d]
    if len(sub) == 0:
        continue
    print(f"{d:>6d} | {sub['a2_resp_cm_per_C'].mean():>+10.2f} | "
          f"{sub['a2_resp_cm_per_C'].std():>10.2f} | {len(sub):>8d}")

# ---------- Which side dominates? ----------
print("\n" + "=" * 100)
print(" WHICH SIDE DOMINATES THE DRIFT?")
print("=" * 100)
print(f"{'direction':>10s} | {'|a1|':>7s} | {'|a2|':>7s} | "
      f"{'dominant':>10s} | {'R²':>6s}")
print("-" * 56)
for _, r in reg_df.iterrows():
    a1_abs = abs(r["a1_init_cm_per_C"])
    a2_abs = abs(r["a2_resp_cm_per_C"])
    if a1_abs + a2_abs < 0.1:
        dom = "none"
    elif a1_abs > 2 * a2_abs:
        dom = "initiator"
    elif a2_abs > 2 * a1_abs:
        dom = "responder"
    else:
        dom = "both"
    print(f"{r['direction']:>10s} | {a1_abs:>7.2f} | {a2_abs:>7.2f} | "
          f"{dom:>10s} | {r['r2']:>6.3f}")

 DUAL TEMPERATURE REGRESSION PER DIRECTED LINK
 Model: bias(t) = a0 + a1·(T_init - T̄_init) + a2·(T_resp - T̄_resp)
 Units: a1, a2 in cm/°C.  bias in cm.
src    link |     n |   a1_init       p |   a2_resp       p |     R² |  corr_init  corr_resp | corr_ii_jj
----------------------------------------------------------------------------------------------------
  1 1-    2 |  3544 |    -28.98       * |     +4.31         |  0.861 |     -0.928     -0.919 |     +0.993
  1 1-    3 |  3585 |    -19.66       * |     +1.79    n.s. |  0.763 |     -0.873     -0.866 |     +0.993
  1 1-    4 |  3585 |    -29.38       * |     +5.08         |  0.849 |     -0.921     -0.914 |     +0.994
  2 1-    2 |  2970 |    -34.54       * |    +64.44       * |  0.927 |     +0.913     +0.949 |     +0.988
  2 2-    3 |  2972 |     +4.44       * |     +7.84       * |  0.971 |     +0.983     +0.984 |     +0.993
  2 2-    4 |  2972 |     -1.30       * |     +1.25       * |  0.012 |     -0.078     -0.068 |     +0.992
  3

In [10]:
# ============================================================
# Cell 7 — Reframed regression: common-mode + differential
# ============================================================
# bias(t) = b0 + b1*(T_avg - T_ref) + b2*(T_i - T_j)
# where T_avg = (T_i + T_j)/2, T_diff = T_i - T_j
# This avoids the multicollinearity problem of the previous fit.
# ============================================================
from scipy.stats import t as t_dist

def ols_full(X, y):
    Xc = np.hstack([np.ones((len(X), 1)), X])
    beta, *_ = np.linalg.lstsq(Xc, y, rcond=None)
    resid = y - Xc @ beta
    n, p = Xc.shape
    dof = max(n - p, 1)
    sigma2 = np.sum(resid ** 2) / dof
    try:
        cov = sigma2 * np.linalg.inv(Xc.T @ Xc)
        se = np.sqrt(np.diag(cov))
    except np.linalg.LinAlgError:
        se = np.full(p, np.nan)
    t_stat = np.where(se > 0, beta / se, 0.0)
    pval = 2 * (1 - t_dist.cdf(np.abs(t_stat), df=dof))
    ss_res = np.sum(resid ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
    return beta, se, pval, r2

print("=" * 108)
print(" REFRAMED REGRESSION  bias = b0 + b1·ΔT_avg + b2·ΔT_diff")
print(" Units: b1, b2 in cm/°C.   ΔT_avg = (T_i+T_j)/2 - mean,  ΔT_diff = T_i - T_j")
print("=" * 108)
print(f"{'dir':>6s} | {'n':>5s} | {'b1 (common)':>13s} {'p':>6s} | "
      f"{'b2 (diff)':>13s} {'p':>6s} | {'R²':>6s} | "
      f"{'|b1|':>7s} | {'|b2|':>7s}")
print("-" * 108)

reframed = []
for src in sorted(df["source_robot"].unique()):
    sub = df[df["source_robot"] == src].copy()
    if len(sub) < 100:
        continue
    for (init, resp), g in sub.groupby(["init_id", "resp_id"]):
        pair = tuple(sorted([str(init), str(resp)]))
        if pair not in GT_pairs:
            continue
        g = g.sort_values("t").reset_index(drop=True)
        
        bias = (g["dist_cfo_m"].values - GT_pairs[pair]) * 100.0
        T_i = g["temp_uwb_init"].values
        T_j = g["temp_uwb_resp"].values
        
        T_avg = 0.5 * (T_i + T_j)
        T_diff = T_i - T_j
        
        T_avg_c = T_avg - T_avg.mean()
        T_diff_c = T_diff - T_diff.mean()
        
        X = np.column_stack([T_avg_c, T_diff_c])
        beta, se, pval, r2 = ols_full(X, bias)
        b0, b1, b2 = beta
        p1, p2 = pval[1], pval[2]
        
        s1 = "*" if p1 < 0.01 else ("." if p1 < 0.05 else "n.s.")
        s2 = "*" if p2 < 0.01 else ("." if p2 < 0.05 else "n.s.")
        
        print(f"{init:>3d}->{resp:<3d}| {len(g):>5d} | "
              f"{b1:>+13.2f} {s1:>6s} | "
              f"{b2:>+13.2f} {s2:>6s} | "
              f"{r2:>6.3f} | "
              f"{abs(b1):>7.2f} | {abs(b2):>7.2f}")
        
        reframed.append({
            "direction": f"{init}->{resp}",
            "init": init, "resp": resp,
            "n": len(g),
            "b1_common_cm_per_C": b1,
            "b1_pvalue": p1,
            "b2_diff_cm_per_C": b2,
            "b2_pvalue": p2,
            "r2": r2,
            "b1_se": se[1],
            "b2_se": se[2],
        })

reframed_df = pd.DataFrame(reframed)
reframed_df.to_csv("output_reframed_thermal.csv", index=False)
print(f"\n  Saved: output_reframed_thermal.csv")

# ---------- Interpretation ----------
print("\n" + "=" * 108)
print(" INTERPRETATION")
print("=" * 108)

# For each direction, determine which term dominates
print(f"\n{'direction':>10s} | {'dominant effect':>20s} | {'b1':>8s} {'b2':>8s}")
print("-" * 58)
for _, r in reframed_df.iterrows():
    b1, b2 = r["b1_common_cm_per_C"], r["b2_diff_cm_per_C"]
    if abs(b1) > 2 * abs(b2):
        dom = "common (both heat)"
    elif abs(b2) > 2 * abs(b1):
        dom = "differential (one side)"
    else:
        dom = "mixed"
    print(f"{r['direction']:>10s} | {dom:>20s} | {b1:>+8.2f} {b2:>+8.2f}")

# Correlation of the two regressors
print(f"\n Mean |corr(T_avg, T_diff)| across links: ", end="")
corrs = []
for src in sorted(df["source_robot"].unique()):
    sub = df[df["source_robot"] == src]
    if len(sub) < 100: continue
    for (init, resp), g in sub.groupby(["init_id", "resp_id"]):
        T_avg = 0.5 * (g["temp_uwb_init"].values + g["temp_uwb_resp"].values)
        T_diff = g["temp_uwb_init"].values - g["temp_uwb_resp"].values
        corrs.append(abs(np.corrcoef(T_avg, T_diff)[0, 1]))
print(f"{np.mean(corrs):.3f}")
print(" (Low value confirms the reframing resolved multicollinearity.)")

 REFRAMED REGRESSION  bias = b0 + b1·ΔT_avg + b2·ΔT_diff
 Units: b1, b2 in cm/°C.   ΔT_avg = (T_i+T_j)/2 - mean,  ΔT_diff = T_i - T_j
   dir |     n |   b1 (common)      p |     b2 (diff)      p |     R² |    |b1| |    |b2|
------------------------------------------------------------------------------------------------------------
  1->2  |  3544 |        -24.66      * |        -16.64      * |  0.861 |   24.66 |   16.64
  1->3  |  3585 |        -17.87      * |        -10.72      * |  0.763 |   17.87 |   10.72
  1->4  |  3585 |        -24.30      * |        -17.23      * |  0.849 |   24.30 |   17.23
  2->1  |  2970 |        +29.90      * |        -49.49      * |  0.927 |   29.90 |   49.49
  2->3  |  2972 |        +12.27      * |         -1.70      * |  0.971 |   12.27 |    1.70
  2->4  |  2972 |         -0.05   n.s. |         -1.28      * |  0.012 |    0.05 |    1.28
  3->1  |  3146 |         +7.74      * |        -36.01      * |  0.916 |    7.74 |   36.01
  3->2  |  3152 |        -12.6

In [11]:
# ============================================================
# Cell 7 — Validate per-direction thermal model
# ============================================================
# For each directed link, apply:
#   bias_pred = b0 + b1*ΔT_avg + b2*ΔT_diff
# Then correct:  d_corrected = d_cfo - bias_pred
# Compute residuals vs GT.
# ============================================================

print("=" * 96)
print(" PER-DIRECTION THERMAL MODEL VALIDATION")
print("=" * 96)

from scipy.stats import t as t_dist

def fit_per_direction(g, gt_val):
    bias = (g["dist_cfo_m"].values - gt_val) * 100.0
    T_avg = 0.5 * (g["temp_uwb_init"].values + g["temp_uwb_resp"].values)
    T_diff = g["temp_uwb_init"].values - g["temp_uwb_resp"].values
    T_avg_c = T_avg - T_avg.mean()
    T_diff_c = T_diff - T_diff.mean()
    X = np.column_stack([np.ones(len(bias)), T_avg_c, T_diff_c])
    beta, *_ = np.linalg.lstsq(X, bias, rcond=None)
    pred = X @ beta
    resid = bias - pred
    r2 = 1 - np.sum(resid**2) / np.sum((bias - bias.mean())**2)
    return beta, pred, resid, r2, T_avg.mean(), T_diff.mean()


results = []
for src in sorted(df["source_robot"].unique()):
    sub = df[df["source_robot"] == src]
    if len(sub) < 100:
        continue
    for (init, resp), g in sub.groupby(["init_id", "resp_id"]):
        pair = tuple(sorted([str(init), str(resp)]))
        if pair not in GT_pairs:
            continue
        g = g.sort_values("t").reset_index(drop=True)
        beta, pred, resid, r2, T_avg_m, T_diff_m = fit_per_direction(g, GT_pairs[pair])
        results.append({
            "dir": f"{init}->{resp}",
            "init": init, "resp": resp,
            "n": len(g),
            "b0": beta[0], "b1": beta[1], "b2": beta[2],
            "r2": r2,
            "rmse_before_cm": np.std((g["dist_cfo_m"].values - GT_pairs[pair]) * 100),
            "rmse_after_cm": np.std(resid),
        })

res_df = pd.DataFrame(results)

print(f"\n{'direction':>10s} | {'n':>5s} | {'RMSE_before':>11s} | "
      f"{'RMSE_after':>10s} | {'improve':>8s} | {'R²':>6s}")
print("-" * 68)
for _, r in res_df.iterrows():
    imp = 100 * (r["rmse_before_cm"] - r["rmse_after_cm"]) / r["rmse_before_cm"]
    print(f"{r['dir']:>10s} | {r['n']:>5d} | {r['rmse_before_cm']:>10.2f}cm | "
          f"{r['rmse_after_cm']:>9.2f}cm | {imp:>7.1f}% | {r['r2']:>6.3f}")

print("-" * 68)
overall_before = np.sqrt(np.mean(res_df["rmse_before_cm"]**2))
overall_after = np.sqrt(np.mean(res_df["rmse_after_cm"]**2))
print(f"{'OVERALL':>10s} |       | {overall_before:>10.2f}cm | "
      f"{overall_after:>9.2f}cm | "
      f"{100*(overall_before-overall_after)/overall_before:>7.1f}% |")

# ---------- Save coefficients ----------
coef_out = res_df[["dir", "init", "resp", "b0", "b1", "b2", "r2"]].copy()
coef_out.columns = ["direction", "init_id", "resp_id",
                    "b0_cm", "b1_cm_per_C", "b2_cm_per_C", "r2"]
coef_out.to_csv("output_per_direction_thermal.csv", index=False)
print(f"\n  Saved: output_per_direction_thermal.csv")

# ---------- Residual ACF ----------
print("\n" + "=" * 96)
print(" RESIDUAL ACF (lag 1, 5, 20)")
print("=" * 96)
def acf_at(x, lag):
    x = x - x.mean()
    denom = np.sum(x * x)
    return float(np.sum(x[:-lag] * x[lag:]) / denom) if denom > 0 else 0.0

print(f"\n{'direction':>10s} | {'ACF[1]':>8s} | {'ACF[5]':>8s} | {'ACF[20]':>8s}")
print("-" * 46)

all_resid = []
for src in sorted(df["source_robot"].unique()):
    sub = df[df["source_robot"] == src]
    if len(sub) < 100:
        continue
    for (init, resp), g in sub.groupby(["init_id", "resp_id"]):
        pair = tuple(sorted([str(init), str(resp)]))
        if pair not in GT_pairs:
            continue
        g = g.sort_values("t").reset_index(drop=True)
        beta, _, resid, _, _, _ = fit_per_direction(g, GT_pairs[pair])
        all_resid.append(resid)
        print(f"{init:>3d}->{resp:<3d}  | {acf_at(resid,1):>8.4f} | "
              f"{acf_at(resid,5):>8.4f} | {acf_at(resid,20):>8.4f}")

all_resid = np.concatenate(all_resid)
print(f"\n  Overall residual σ: {np.std(all_resid):.2f} cm")
print(f"  Overall ACF[1]: {acf_at(all_resid, 1):+.4f}  "
      f"(<0.3 = white, >0.7 = strong structure)")

 PER-DIRECTION THERMAL MODEL VALIDATION

 direction |     n | RMSE_before | RMSE_after |  improve |     R²
--------------------------------------------------------------------
      1->2 |  3544 |     139.35cm |     52.01cm |    62.7% |  0.861
      1->3 |  3585 |     104.77cm |     51.06cm |    51.3% |  0.763
      1->4 |  3585 |     138.31cm |     53.73cm |    61.2% |  0.849
      2->1 |  2970 |     138.71cm |     37.55cm |    72.9% |  0.927
      2->3 |  2972 |      50.33cm |      8.59cm |    82.9% |  0.971
      2->4 |  2972 |       7.68cm |      7.63cm |     0.6% |  0.012
      3->1 |  3146 |      74.18cm |     21.49cm |    71.0% |  0.916
      3->2 |  3152 |      44.79cm |      7.58cm |    83.1% |  0.971
      3->4 |  3151 |      46.63cm |      5.19cm |    88.9% |  0.988
      4->1 |  3407 |     135.91cm |     45.17cm |    66.8% |  0.890
      4->2 |  3420 |       9.07cm |      8.22cm |     9.4% |  0.178
      4->3 |  3418 |      45.64cm |      5.78cm |    87.3% |  0.984
--------

In [12]:
# ============================================================
# Cell 8 — Characterize leftover temporal structure
# ============================================================
# After thermal correction, residuals still have ACF ~0.95.
# Two hypotheses:
#   H1: Common per-robot drift — residuals of links from same
#       source robot move together
#   H2: Per-link drift — each link has independent drift
# ============================================================

print("=" * 96)
print(" TEST 1: Is residual drift COMMON across links from same source robot?")
print("=" * 96)
print(" For each source robot, compute correlation of residuals across")
print(" its three outgoing links (time-aligned by frame).")
print("-" * 96)

# Collect residuals per (source, link) with frame alignment
resid_dict = {}  # key: (src, init, resp) -> DataFrame(frame_id, t, resid)

for src in sorted(df["source_robot"].unique()):
    sub = df[df["source_robot"] == src]
    if len(sub) < 100:
        continue
    for (init, resp), g in sub.groupby(["init_id", "resp_id"]):
        pair = tuple(sorted([str(init), str(resp)]))
        if pair not in GT_pairs:
            continue
        g = g.sort_values("frame_id").reset_index(drop=True)
        beta, _, resid, _, _, _ = fit_per_direction(g, GT_pairs[pair])
        resid_dict[(src, init, resp)] = pd.DataFrame({
            "frame_id": g["frame_id"].values,
            "t": g["t"].values,
            "resid_cm": resid,
        })

# Now, for each source robot, compute pairwise correlation of residuals
print(f"\n{'source':>6s} | {'link A':>8s} | {'link B':>8s} | "
      f"{'n_overlap':>9s} | {'corr':>8s}")
print("-" * 56)

for src in sorted(df["source_robot"].unique()):
    keys = [k for k in resid_dict.keys() if k[0] == src]
    if len(keys) < 2:
        continue
    for i in range(len(keys)):
        for j in range(i+1, len(keys)):
            kA, kB = keys[i], keys[j]
            dfA = resid_dict[kA].set_index("frame_id")
            dfB = resid_dict[kB].set_index("frame_id")
            common = dfA.index.intersection(dfB.index)
            if len(common) < 50:
                continue
            rA = dfA.loc[common, "resid_cm"].values
            rB = dfB.loc[common, "resid_cm"].values
            # Remove mean
            rA_c = rA - rA.mean()
            rB_c = rB - rB.mean()
            corr = np.corrcoef(rA_c, rB_c)[0, 1]
            print(f"{src:>6d} | {kA[1]}->{kA[2]:>4d} | {kB[1]}->{kB[2]:>4d} | "
                  f"{len(common):>9d} | {corr:>+8.3f}")


# ============================================================
# TEST 2: Does residual correlate with time?
# ============================================================
print("\n" + "=" * 96)
print(" TEST 2: Residual vs time — is there pure time drift left?")
print("=" * 96)
print(f"{'direction':>10s} | {'corr(resid, t)':>15s} | "
      f"{'slope (cm/min)':>15s} | {'R²':>6s}")
print("-" * 60)

for src in sorted(df["source_robot"].unique()):
    sub = df[df["source_robot"] == src]
    if len(sub) < 100:
        continue
    for (init, resp), g in sub.groupby(["init_id", "resp_id"]):
        pair = tuple(sorted([str(init), str(resp)]))
        if pair not in GT_pairs:
            continue
        g = g.sort_values("t").reset_index(drop=True)
        beta, _, resid, _, _, _ = fit_per_direction(g, GT_pairs[pair])
        t = g["t"].values
        # Linear fit of resid vs t
        t_c = t - t.mean()
        X = np.column_stack([np.ones(len(t)), t_c])
        coef, *_ = np.linalg.lstsq(X, resid, rcond=None)
        pred = X @ coef
        ss_res = np.sum((resid - pred) ** 2)
        ss_tot = np.sum((resid - resid.mean()) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0
        corr = np.corrcoef(resid, t)[0, 1]
        slope_per_min = coef[1] * 60.0  # cm per minute
        print(f"{init:>3d}->{resp:<3d}  | {corr:>+15.4f} | "
              f"{slope_per_min:>+15.3f} | {r2:>6.3f}")


# ============================================================
# TEST 3: Residual vs temperature (after linear removal)
# ============================================================
print("\n" + "=" * 96)
print(" TEST 3: Residual vs temperature (nonlinearity check)")
print("=" * 96)
print(" If linear thermal model captured everything, this correlation should be ~0.")
print(f"{'direction':>10s} | {'corr(resid, T_avg)':>18s} | "
      f"{'corr(resid, T_diff)':>19s}")
print("-" * 58)

for src in sorted(df["source_robot"].unique()):
    sub = df[df["source_robot"] == src]
    if len(sub) < 100:
        continue
    for (init, resp), g in sub.groupby(["init_id", "resp_id"]):
        pair = tuple(sorted([str(init), str(resp)]))
        if pair not in GT_pairs:
            continue
        g = g.sort_values("t").reset_index(drop=True)
        beta, _, resid, _, _, _ = fit_per_direction(g, GT_pairs[pair])
        T_avg = 0.5 * (g["temp_uwb_init"].values + g["temp_uwb_resp"].values)
        T_diff = g["temp_uwb_init"].values - g["temp_uwb_resp"].values
        c1 = np.corrcoef(resid, T_avg)[0, 1]
        c2 = np.corrcoef(resid, T_diff)[0, 1]
        print(f"{init:>3d}->{resp:<3d}  | {c1:>+18.4f} | {c2:>+19.4f}")


# ============================================================
# TEST 4: Save residuals for further analysis
# ============================================================
all_resid_rows = []
for src in sorted(df["source_robot"].unique()):
    sub = df[df["source_robot"] == src]
    if len(sub) < 100:
        continue
    for (init, resp), g in sub.groupby(["init_id", "resp_id"]):
        pair = tuple(sorted([str(init), str(resp)]))
        if pair not in GT_pairs:
            continue
        g = g.sort_values("t").reset_index(drop=True)
        beta, _, resid, _, _, _ = fit_per_direction(g, GT_pairs[pair])
        for k in range(len(g)):
            all_resid_rows.append({
                "source": src, "init": init, "resp": resp,
                "direction": f"{init}->{resp}",
                "frame_id": int(g["frame_id"].iloc[k]),
                "t": g["t"].iloc[k],
                "T_init": g["temp_uwb_init"].iloc[k],
                "T_resp": g["temp_uwb_resp"].iloc[k],
                "resid_cm": resid[k],
            })

resid_full = pd.DataFrame(all_resid_rows)
resid_full.to_csv("output_residuals_after_thermal.csv", index=False)
print(f"\n  Saved: output_residuals_after_thermal.csv  ({len(resid_full)} rows)")

 TEST 1: Is residual drift COMMON across links from same source robot?
 For each source robot, compute correlation of residuals across
 its three outgoing links (time-aligned by frame).
------------------------------------------------------------------------------------------------

source |   link A |   link B | n_overlap |     corr
--------------------------------------------------------
     1 | 1->   2 | 1->   3 |      3544 |   +0.960
     1 | 1->   2 | 1->   4 |      3544 |   +0.985
     1 | 1->   3 | 1->   4 |      3585 |   +0.987
     2 | 2->   1 | 2->   3 |      2970 |   -0.020
     2 | 2->   1 | 2->   4 |      2970 |   -0.050
     2 | 2->   3 | 2->   4 |      2972 |   +0.823
     3 | 3->   1 | 3->   2 |      3146 |   +0.369
     3 | 3->   1 | 3->   4 |      3145 |   +0.093
     3 | 3->   2 | 3->   4 |      3151 |   +0.315
     4 | 4->   1 | 4->   2 |      3407 |   +0.378
     4 | 4->   1 | 4->   3 |      3405 |   +0.508
     4 | 4->   2 | 4->   3 |      3418 |   +0.232

 TEST 

In [13]:
# ============================================================
# Cell 9 — Test thermal lag and ESP temperature effects
# ============================================================
# Hypothesis: The residual structure comes from
#   (a) thermal lag (crystal temp ≠ chip temp)
#   (b) ESP32 temperature effect on timing
#   (c) quadratic thermal response
# ============================================================

# --- Helper: first-order low-pass filter ---
def apply_lpf(sig, t, tau):
    """y[k] = y[k-1] + (dt/τ)·(x[k] - y[k-1])"""
    y = np.zeros_like(sig, dtype=float)
    y[0] = sig[0]
    for k in range(1, len(sig)):
        dt = t[k] - t[k-1]
        if dt <= 0: dt = 0.2
        a = dt / (tau + dt)
        y[k] = y[k-1] + a * (sig[k] - y[k-1])
    return y


print("=" * 100)
print(" TEST 4: ADD LAGGED TEMPERATURE TO MODEL")
print("=" * 100)
print(" Model: bias = b0 + b1·T_avg_filt + b2·T_diff_filt + b3·T_esp_avg")
print(" Grid search on τ (thermal lag time constant).")
print("-" * 100)
print(f"{'τ[s]':>6s} | {'mean R²':>10s} | {'mean resid σ':>14s} | "
      f"{'ACF[1] (avg)':>14s}")
print("-" * 66)

TAU_GRID = [1, 5, 10, 20, 30, 60, 120, 180]

for tau in TAU_GRID:
    r2_list = []
    resid_list = []
    acf_list = []
    for src in sorted(df["source_robot"].unique()):
        sub = df[df["source_robot"] == src]
        if len(sub) < 100: continue
        for (init, resp), g in sub.groupby(["init_id", "resp_id"]):
            pair = tuple(sorted([str(init), str(resp)]))
            if pair not in GT_pairs: continue
            g = g.sort_values("t").reset_index(drop=True)
            bias = (g["dist_cfo_m"].values - GT_pairs[pair]) * 100.0
            t = g["t"].values
            
            # Chip temps
            T_i = g["temp_uwb_init"].values
            T_j = g["temp_uwb_resp"].values
            # Filter
            T_i_f = apply_lpf(T_i, t, tau)
            T_j_f = apply_lpf(T_j, t, tau)
            
            T_avg = 0.5 * (T_i_f + T_j_f)
            T_diff = T_i_f - T_j_f
            T_esp_avg = 0.5 * (g["temp_esp_init"].values
                                + g["temp_esp_resp"].values)
            
            X = np.column_stack([T_avg, T_diff, T_esp_avg])
            Xc = np.hstack([np.ones((len(X), 1)), X])
            beta, *_ = np.linalg.lstsq(Xc, bias, rcond=None)
            resid = bias - Xc @ beta
            ss_res = np.sum(resid**2)
            ss_tot = np.sum((bias - bias.mean())**2)
            r2_list.append(1 - ss_res/ss_tot if ss_tot > 0 else 0)
            resid_list.append(resid)
            x = resid - resid.mean()
            acf_list.append(np.sum(x[:-1]*x[1:]) / np.sum(x*x))
    
    mean_r2 = np.mean(r2_list)
    mean_std = np.mean([np.std(r) for r in resid_list])
    mean_acf = np.mean(acf_list)
    print(f"{tau:>6.1f} | {mean_r2:>10.4f} | {mean_std:>13.2f}cm | "
          f"{mean_acf:>+14.4f}")


print("\n" + "=" * 100)
print(" TEST 5: PER-LINK τ OPTIMIZATION")
print("=" * 100)
print(" For each direction, find τ that minimizes residual ACF[1].")
print("-" * 100)
print(f"{'direction':>10s} | {'best_τ':>8s} | {'R²':>6s} | "
      f"{'resid σ':>10s} | {'ACF[1]':>8s}")
print("-" * 60)

tau_results = []
for src in sorted(df["source_robot"].unique()):
    sub = df[df["source_robot"] == src]
    if len(sub) < 100: continue
    for (init, resp), g in sub.groupby(["init_id", "resp_id"]):
        pair = tuple(sorted([str(init), str(resp)]))
        if pair not in GT_pairs: continue
        g = g.sort_values("t").reset_index(drop=True)
        bias = (g["dist_cfo_m"].values - GT_pairs[pair]) * 100.0
        t = g["t"].values
        T_i = g["temp_uwb_init"].values
        T_j = g["temp_uwb_resp"].values
        
        best = None
        for tau in TAU_GRID:
            T_i_f = apply_lpf(T_i, t, tau)
            T_j_f = apply_lpf(T_j, t, tau)
            T_avg = 0.5 * (T_i_f + T_j_f)
            T_diff = T_i_f - T_j_f
            Xc = np.column_stack([np.ones(len(bias)), T_avg, T_diff])
            beta, *_ = np.linalg.lstsq(Xc, bias, rcond=None)
            resid = bias - Xc @ beta
            ss_res = np.sum(resid**2)
            ss_tot = np.sum((bias - bias.mean())**2)
            r2 = 1 - ss_res/ss_tot if ss_tot > 0 else 0
            x = resid - resid.mean()
            acf1 = np.sum(x[:-1]*x[1:]) / np.sum(x*x)
            if best is None or abs(acf1) < abs(best["acf1"]):
                best = {"tau": tau, "r2": r2, "std": np.std(resid),
                        "acf1": acf1}
        
        print(f"{init:>3d}->{resp:<3d}  | {best['tau']:>8.1f} | "
              f"{best['r2']:>6.3f} | {best['std']:>9.2f}cm | "
              f"{best['acf1']:>+8.4f}")
        tau_results.append({
            "direction": f"{init}->{resp}",
            "init": init, "resp": resp,
            "best_tau_s": best["tau"],
            "r2": best["r2"],
            "resid_std_cm": best["std"],
            "acf1": best["acf1"],
        })

tau_df = pd.DataFrame(tau_results)
tau_df.to_csv("output_per_link_tau.csv", index=False)
print(f"\n  Saved: output_per_link_tau.csv")

 TEST 4: ADD LAGGED TEMPERATURE TO MODEL
 Model: bias = b0 + b1·T_avg_filt + b2·T_diff_filt + b3·T_esp_avg
 Grid search on τ (thermal lag time constant).
----------------------------------------------------------------------------------------------------
  τ[s] |    mean R² |   mean resid σ |   ACF[1] (avg)
------------------------------------------------------------------
   1.0 |     0.7808 |         24.48cm |        +0.9524
   5.0 |     0.7880 |         23.49cm |        +0.9469
  10.0 |     0.7981 |         22.65cm |        +0.9388
  20.0 |     0.8083 |         21.35cm |        +0.9314
  30.0 |     0.8123 |         20.16cm |        +0.9280
  60.0 |     0.8267 |         17.62cm |        +0.9198
 120.0 |     0.8342 |         16.19cm |        +0.9157
 180.0 |     0.8339 |         16.05cm |        +0.9161

 TEST 5: PER-LINK τ OPTIMIZATION
 For each direction, find τ that minimizes residual ACF[1].
------------------------------------------------------------------------------------------

In [14]:
# ============================================================
# Cell 10 — Test whether residual behaves as random walk
# ============================================================
# Key property of a random walk: variance grows linearly with 
# time-lag. If we see Var(resid(t+τ) - resid(t)) ∝ τ, then 
# the bias has a true random walk component.
# ============================================================

print("=" * 100)
print(" VARIANCE-GROWTH TEST  (random walk detection)")
print("=" * 100)
print(" For each direction: compute Var[resid(t+τ) - resid(t)]")
print(" If linear in τ → random walk. If saturating → mean-reverting.")
print("-" * 100)

LAGS_SEC = [5, 10, 20, 40, 80, 160, 300]
print(f"{'direction':>10s} | " + " ".join(f"{l:>6d}" for l in LAGS_SEC))
print("-" * 100)

# Use residuals already saved
resid_full = pd.read_csv("output_residuals_after_thermal.csv")

for direction in sorted(resid_full["direction"].unique()):
    sub = resid_full[resid_full["direction"] == direction].sort_values("t")
    t = sub["t"].values
    r = sub["resid_cm"].values
    if len(r) < 100:
        continue
    var_row = []
    for lag_s in LAGS_SEC:
        # Find pairs with time-lag ≈ lag_s
        # Use frame spacing ~0.2s, so lag_s/0.2 frames
        step = max(1, int(lag_s / 0.2))
        if step >= len(r):
            var_row.append(np.nan)
            continue
        diff = r[step:] - r[:-step]
        var_row.append(np.var(diff))
    print(f"{direction:>10s} | " + " ".join(
        f"{v:>6.1f}" if not np.isnan(v) else f"{'--':>6s}" for v in var_row))

print("\n  If row is roughly LINEAR in lag → random walk behavior.")
print("  If row saturates → bounded / mean-reverting.")


print("\n" + "=" * 100)
print(" ACF DECAY COMPARISON")
print("=" * 100)
print(" Compare ACF shape after simple thermal vs after lag-thermal.")
print("-" * 100)
print(f"{'direction':>10s} | " + " ".join(f"{l:>6d}" for l in [1,2,5,10,20,50]))
print("-" * 100)

for direction in sorted(resid_full["direction"].unique()):
    sub = resid_full[resid_full["direction"] == direction].sort_values("t")
    r = sub["resid_cm"].values - sub["resid_cm"].mean()
    n = len(r)
    denom = np.sum(r*r)
    acf_row = []
    for lag in [1,2,5,10,20,50]:
        if lag >= n:
            acf_row.append(np.nan); continue
        acf_row.append(np.sum(r[:-lag]*r[lag:]) / denom)
    print(f"{direction:>10s} | " + " ".join(
        f"{v:>6.3f}" for v in acf_row))


print("\n" + "=" * 100)
print(" SAVED RESIDUAL SUMMARY")
print("=" * 100)
print(f"{'direction':>10s} | {'n':>6s} | {'mean':>8s} | "
      f"{'std':>8s} | {'range':>8s}")
print("-" * 56)
for direction in sorted(resid_full["direction"].unique()):
    sub = resid_full[resid_full["direction"] == direction]
    print(f"{direction:>10s} | {len(sub):>6d} | {sub['resid_cm'].mean():>+8.2f} | "
          f"{sub['resid_cm'].std():>8.2f} | "
          f"{sub['resid_cm'].max()-sub['resid_cm'].min():>8.2f}")

print("\n" + "=" * 100)
print(" INTERPRETATION")
print("=" * 100)
print("""
Three possible interpretations:

A) Residual is a RANDOM WALK:
   → Variance grows linearly with lag
   → Requires EKF with bias state and process noise Q

B) Residual is MEAN-REVERTING (Ornstein-Uhlenbeck):
   → Variance saturates at 2·σ²
   → Requires EKF with bias state and damping

C) Residual is DETERMINISTIC but non-thermal:
   → Variance bounded and ACF does not decay smoothly
   → Requires physical investigation (PCB, antenna, aging)

Look at the variance-growth table:
- If Var grows ~linearly with lag → case A
- If Var saturates at some value → case B
- If Var is erratic → case C
""")

 VARIANCE-GROWTH TEST  (random walk detection)
 For each direction: compute Var[resid(t+τ) - resid(t)]
 If linear in τ → random walk. If saturating → mean-reverting.
----------------------------------------------------------------------------------------------------
 direction |      5     10     20     40     80    160    300
----------------------------------------------------------------------------------------------------
      1->2 |  142.5  224.8  399.2  818.3 2372.3 6941.6 9089.7
      1->3 |   75.3  128.0  290.5  835.4 2552.0 6628.2 8451.2
      1->4 |  137.2  201.3  361.3  883.0 2613.0 7447.8 9878.8
      2->1 |  652.9  801.3 1073.1 1467.7 2177.9 3657.7 2568.1
      2->3 |   33.6   66.1  133.1  185.7   90.2  110.4  168.8
      2->4 |   22.5   53.9  110.8  149.5   74.5  107.2  115.8
      3->1 |  340.3  399.8  491.0  621.3  931.5 1086.4  781.7
      3->2 |   33.3   61.6  101.5  100.7   81.9  109.3  109.7
      3->4 |   25.4   31.3   35.9   41.9   44.6   55.3   71.2
      4->1 |